# Cell Tracking Baseline: Adaptive Per-Movie Node Budgets

This is a from-scratch baseline for the embryo cell-tracking task: turn each 3D+time
movie into a lineage graph (nodes = detected cell centers per frame, edges = links
between consecutive frames).

**Core idea.** The scoring metric punishes over-predicting the *total* node count, but
the true number of cells per embryo varies a lot (some movies have a handful of cells
per frame, others well over a hundred). A single global detection threshold will always
be wrong for most movies, too loose on sparse ones, too tight on dense ones.

So this notebook splits detection into two stages:

1. **Detect generously**: use a low bar so we rarely miss a real cell.
2. **Calibrate a per-movie budget**: learn, from the training movies (which ship
   ground-truth counts), a cheap mapping from simple image statistics to an expected
   cell count, and keep only the top-ranked candidates up to that budget on each
   test movie.

Everything else (linking, pruning) stays deliberately simple: a tight µm-space
distance gate for frame-to-frame matching, no division detection to start with
(mitoses are rare and a false division costs two mistakes at once), and dropping
detections that never link to anything, since those are almost always noise.


# 1. Setup and configuration

In [ ]:
!pip install --no-index --find-links /kaggle/input/datasets/sarveshchhetri/zarr-offline-wheel/wheels/kaggle/working/wheels zarr

In [ ]:
import os
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

try:
    import zarr
except ImportError as e:
    raise ImportError(
        'zarr is required to read the competition data. Run `!pip install zarr` '
        'in a cell above this one, then restart the kernel and re-run.'
    ) from e

RNG = np.random.default_rng(0)

In [ ]:
# physical geometry 
VOXEL_UM = np.array([1.625, 0.40625, 0.40625])   # (z, y, x) microns per voxel
GT_MATCH_RADIUS_UM = 7.0                          # how the scorer matches predictions to GT

# detection 
XY_BIN          = 4        # spatial downsample in y/x before peak search
SMOOTH_SIGMA    = 1.0      # gaussian pre-smoothing (in downsampled voxels)
PEAK_MIN_SEP    = 2        # min separation between raw peaks, in downsampled voxels
NMS_RADIUS_UM   = 4.0      # physical non-max-suppression radius
LOW_THRESH_Q    = 0.55     # quantile cut for the *generous* candidate pool (low bar)
TIGHT_THRESH_Q  = 0.80     # quantile cut used when calibration is turned off

# linking 
LINK_GATE_UM   = 10.0      # max physical distance allowed between linked frames
USE_VELOCITY   = True      # extrapolate last displacement before gating the next frame

# structural choices 
ENABLE_DIVISIONS = False   # keep off: rare event, expensive false positives
DROP_ORPHANS     = True    # drop nodes that end up with zero edges

# per-movie count calibration
ENABLE_CALIBRATION   = True
CALIB_FRAMES_PER_MOVIE = 8    # frames sampled per training movie to fit the calibrator
BUDGET_MARGIN         = 1.10  # predicted_count * margin = kept candidates (recall cushion)

SUBMISSION_COLUMNS = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']

# 2. Locate the data

In [ ]:
def locate_competition_root():
    """Find the train/ and test/ directories regardless of how the dataset was mounted."""
    candidates = [
        Path('/kaggle/input/biohub-cell-tracking-during-development'),
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
    ]
    for c in candidates:
        if (c / 'test').is_dir():
            return c / 'train', c / 'test'
    # fall back to a search
    hits = glob.glob('/kaggle/input/**/test', recursive=True)
    if hits:
        root = Path(hits[0]).parent
        return root / 'train', root / 'test'
    raise FileNotFoundError('Could not locate competition data under /kaggle/input')

TRAIN_DIR, TEST_DIR = locate_competition_root()
print(f'train: {TRAIN_DIR} (exists={TRAIN_DIR.is_dir()})')
print(f'test:  {TEST_DIR} (exists={TEST_DIR.is_dir()})')

In [ ]:
def movie_names(directory):
    if not directory.is_dir():
        return []
    return sorted(p.stem for p in directory.glob('*.zarr'))

train_movies = movie_names(TRAIN_DIR)
test_movies = movie_names(TEST_DIR)
print(f'{len(train_movies)} train movies, {len(test_movies)} test movies')

# 3. I/O helpers

In [ ]:
_ZARR_CACHE = {}

def open_movie(zarr_path):
    if zarr_path not in _ZARR_CACHE:
        _ZARR_CACHE[zarr_path] = zarr.open(str(zarr_path), mode='r')['0']
    return _ZARR_CACHE[zarr_path]

def read_frame(zarr_path, t):
    arr = open_movie(zarr_path)
    return np.asarray(arr[t])

def num_frames(zarr_path):
    return open_movie(zarr_path).shape[0]

def read_ground_truth(geff_path):
    """Read the training-set lineage graph (GEFF format) into node/edge frames."""
    g = zarr.open_group(str(geff_path), mode='r')
    node_ids = np.asarray(g['nodes/ids'][:]).astype(np.int64)
    coords = {k: np.asarray(g[f'nodes/props/{k}/values'][:]) for k in ('t', 'z', 'y', 'x')}
    nodes = pd.DataFrame({'node_id': node_ids, **coords})

    edge_ids = np.asarray(g['edges/ids'][:])
    edges = pd.DataFrame({'source_id': edge_ids[:, 0].astype(np.int64),
                           'target_id': edge_ids[:, 1].astype(np.int64)})
    return nodes, edges

# 4. Detection
Smooth, downsample in-plane (z stays full resolution since the anisotropy is already
~4x), and pull local maxima. Peaks are kept as candidates with a strength score; how
many candidates survive depends on the threshold quantile (loose for calibration,
tight otherwise), followed by a physical-µm non-max-suppression pass so we don't keep
two peaks that are really the same cell.

In [ ]:
def find_candidate_peaks(volume, threshold_quantile):
    """Return (voxel_coords[z,y,x], scores) for local maxima above a quantile cutoff."""
    small = volume[:, ::XY_BIN, ::XY_BIN].astype(np.float32)
    smoothed = gaussian_filter(small, sigma=SMOOTH_SIGMA)

    footprint_size = (max(1, PEAK_MIN_SEP // 2), PEAK_MIN_SEP, PEAK_MIN_SEP)
    local_max = maximum_filter(smoothed, size=footprint_size) == smoothed

    nonzero = smoothed[smoothed > 0]
    if nonzero.size == 0:
        return np.empty((0, 3), dtype=np.int64), np.empty((0,), dtype=np.float32)
    cutoff = np.quantile(nonzero, threshold_quantile)

    mask = local_max & (smoothed > cutoff)
    zz, yy, xx = np.nonzero(mask)
    scores = smoothed[zz, yy, xx]

    coords = np.stack([zz, yy * XY_BIN, xx * XY_BIN], axis=1)
    return coords, scores

def suppress_near_duplicates(coords_voxel, scores, radius_um):
    """Greedy NMS in physical space: keep the strongest peak, drop weaker ones nearby."""
    if len(coords_voxel) == 0:
        return coords_voxel, scores
    coords_um = coords_voxel * VOXEL_UM
    order = np.argsort(-scores)
    tree = cKDTree(coords_um)
    keep = np.ones(len(coords_voxel), dtype=bool)
    for i in order:
        if not keep[i]:
            continue
        nearby = tree.query_ball_point(coords_um[i], r=radius_um)
        for j in nearby:
            if j != i and scores[j] <= scores[i]:
                keep[j] = False
    return coords_voxel[keep], scores[keep]

def detect_frame(volume, threshold_quantile):
    coords, scores = find_candidate_peaks(volume, threshold_quantile)
    coords, scores = suppress_near_duplicates(coords, scores, NMS_RADIUS_UM)
    order = np.argsort(-scores)
    return coords[order], scores[order]

# 5. Per-movie count calibration
Training movies come with ground-truth node counts per frame. We compute a couple of
cheap, detector-agnostic image statistics per frame (candidate count at the generous
threshold, and mean smoothed intensity above background) and fit a small linear model
predicting the true count from those statistics. At test time, the same statistics are
computed and fed through the fitted model to get a target node budget for each frame.

In [ ]:
def frame_statistics(volume):
    coords, scores = find_candidate_peaks(volume, LOW_THRESH_Q)
    n_candidates = len(coords)
    mean_strength = float(scores.mean()) if len(scores) else 0.0
    return np.array([1.0, n_candidates, mean_strength])  # bias term + 2 features

def fit_count_calibrator():
    X_rows, y_rows = [], []
    for name in train_movies:
        zp = TRAIN_DIR / f'{name}.zarr'
        geff_path = TRAIN_DIR / f'{name}.geff'
        if not geff_path.exists() or not zp.exists():
            continue
        nodes, _ = read_ground_truth(geff_path)
        true_counts_by_t = nodes.groupby('t').size()

        T = num_frames(zp)
        sample_ts = RNG.choice(T, size=min(CALIB_FRAMES_PER_MOVIE, T), replace=False)
        for t in sample_ts:
            if t not in true_counts_by_t.index:
                continue
            vol = read_frame(zp, int(t))
            X_rows.append(frame_statistics(vol))
            y_rows.append(true_counts_by_t.loc[t])

    if len(X_rows) < 3:
        return None  # not enough signal to fit safely; caller should fall back

    X = np.stack(X_rows)
    y = np.array(y_rows, dtype=np.float64)
    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coeffs

def predict_frame_budget(volume, coeffs):
    if coeffs is None:
        return None
    feats = frame_statistics(volume)
    pred = float(feats @ coeffs)
    return max(1, int(round(pred * BUDGET_MARGIN)))

count_calibrator = fit_count_calibrator() if ENABLE_CALIBRATION else None
print('calibrator fitted:', count_calibrator is not None)
if count_calibrator is not None:
    print('coefficients [bias, n_candidates, mean_strength]:', np.round(count_calibrator, 4))

# 6. Linking
Consecutive frames are matched with a Hungarian assignment restricted to pairs within
`LINK_GATE_UM` of each other in physical space (unmatched detections just start or end
a track segment). Optionally, each track's last known displacement is used to shift its
predicted position before gating the next frame, which helps with any consistent drift.

In [ ]:
def link_frames(prev_um, curr_um, prev_velocity=None):
    """Return list of (i, j) index pairs linking prev -> curr, gated by LINK_GATE_UM."""
    if len(prev_um) == 0 or len(curr_um) == 0:
        return []

    predicted = prev_um + prev_velocity if (USE_VELOCITY and prev_velocity is not None) else prev_um
    cost = np.linalg.norm(predicted[:, None, :] - curr_um[None, :, :], axis=2)
    cost[cost > LINK_GATE_UM] = 1e6  # effectively forbidden

    row_ind, col_ind = linear_sum_assignment(cost)
    pairs = [(int(i), int(j)) for i, j in zip(row_ind, col_ind) if cost[i, j] < 1e6]
    return pairs

# 7. Full per-movie pipeline

In [ ]:
def track_movie(zp, dataset_name, coeffs):
    T = num_frames(zp)
    all_nodes, all_edges = [], []
    next_id = 1
    prev_ids, prev_um, prev_velocity = None, None, None

    for t in range(T):
        vol = read_frame(zp, t)

        if coeffs is not None:
            budget = predict_frame_budget(vol, coeffs)
            coords, scores = detect_frame(vol, LOW_THRESH_Q)
            if budget is not None and len(coords) > budget:
                coords, scores = coords[:budget], scores[:budget]
        else:
            coords, scores = detect_frame(vol, TIGHT_THRESH_Q)

        ids = np.arange(next_id, next_id + len(coords))
        next_id += len(coords)
        coords_um = coords * VOXEL_UM

        for node_id, (z, y, x) in zip(ids, coords):
            all_nodes.append({'dataset': dataset_name, 'row_type': 'node', 'node_id': int(node_id),
                               't': t, 'z': float(z * VOXEL_UM[0]), 'y': float(y * VOXEL_UM[1]),
                               'x': float(x * VOXEL_UM[2]), 'source_id': -1, 'target_id': -1})

        if prev_ids is not None:
            pairs = link_frames(prev_um, coords_um, prev_velocity)
            for i, j in pairs:
                all_edges.append({'dataset': dataset_name, 'row_type': 'edge', 'node_id': -1,
                                   't': -1, 'z': -1, 'y': -1, 'x': -1,
                                   'source_id': int(prev_ids[i]), 'target_id': int(ids[j])})
            if USE_VELOCITY and pairs:
                # velocity must be indexed against coords_um (this frame's nodes),
                # since coords_um becomes prev_um on the next iteration
                matched_prev = np.array([i for i, _ in pairs])
                matched_curr = np.array([j for _, j in pairs])
                prev_velocity = np.zeros_like(coords_um)
                prev_velocity[matched_curr] = coords_um[matched_curr] - prev_um[matched_prev]
            else:
                prev_velocity = None

        prev_ids, prev_um = ids, coords_um

    nodes_df = pd.DataFrame(all_nodes)
    edges_df = pd.DataFrame(all_edges)

    if DROP_ORPHANS and len(nodes_df):
        linked = set(edges_df.source_id) | set(edges_df.target_id)
        nodes_df = nodes_df[nodes_df.node_id.isin(linked)]

    stats = {'name': dataset_name, 'T': T, 'nodes': len(nodes_df), 'edges': len(edges_df),
              'cells_per_frame': len(nodes_df) / T if T else 0.0}
    return nodes_df, edges_df, stats

# 8. Run on the test set and write the submission

In [ ]:
parts = []
run_stats = []

for name in test_movies:
    zp = TEST_DIR / f'{name}.zarr'
    if not (zp / '0' / 'zarr.json').exists():
        print(f'  skipping {name}: no readable metadata')
        continue
    nodes_df, edges_df, stats = track_movie(zp, name, count_calibrator)
    run_stats.append(stats)
    parts += [nodes_df, edges_df]
    print(f"  {name}: T={stats['T']} nodes={stats['nodes']} edges={stats['edges']} "
          f"cells/frame={stats['cells_per_frame']:.1f}")

submission = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=SUBMISSION_COLUMNS)
submission = submission[SUBMISSION_COLUMNS]
submission.index.name = 'id'
submission.to_csv('submission.csv')
print(f'\nwrote submission.csv: {len(submission)} rows')

In [ ]:
# sanity checks
nodes_only = submission[submission.row_type == 'node']
edges_only = submission[submission.row_type == 'edge']

assert (edges_only[['node_id', 't', 'z', 'y', 'x']] == -1).all().all(), 'edge rows must blank out node fields'

for ds, g in submission.groupby('dataset'):
    node_ids = set(g[g.row_type == 'node'].node_id)
    e = g[g.row_type == 'edge']
    assert (set(e.source_id) | set(e.target_id)).issubset(node_ids), f'dangling edge reference in {ds}'
    assert g[g.row_type == 'node'].node_id.is_unique, f'duplicate node_id in {ds}'

print('all checks passed')
pd.DataFrame(run_stats)

# 9 · Where to push from here

- **`BUDGET_MARGIN`** is the single highest-leverage knob which it trades recall against
  the over-prediction penalty. Sweep it on the leaderboard once the pipeline runs end
  to end.
- **`LOW_THRESH_Q`** controls how deep the generous candidate pool goes before the
  budget trims it; pair any change here with a re-check of `BUDGET_MARGIN`.
- **`LINK_GATE_UM`** should be set from the actual displacement distribution in the
  training ground truth (e.g. the 99th percentile of matched frame-to-frame moves)
  rather than guessed.
- **Divisions** are off by default and only turn them on if a validation split with
  known divisions shows a clear net gain, since a false division costs two mistakes
  (an edge error and a division error) at once.
- **The biggest realistic ceiling-raiser** is swapping `find_candidate_peaks` for a
  learned 3D detector (e.g. a pretrained nucleus-segmentation model shipped as a
  Kaggle dataset, since internet access is off). Everything downstream calibration,
  linking, pruning, submission writing — stays the same; only the detection function
  changes.
